# Step 1: Installation of RDKit 
- RDKit is a collection of open-source cheminformatics and machine-learning software written primarily in C++ with accessible Python wrappers. It is widely used in drug discovery and computational chemistry to handle chemical data, calculate molecular descriptors, perform substructure searches, and visualize 2D or 3D molecular structures. 
## Core Features and CapabilitiesMolecular Representation: 
- Read, write, and convert formats like SMILES, SMARTS, SDF, and PDB.Calculations: Compute molecular weights, LogP, topological polar surface area, fingerprints (Molecular fingerprints are fixed-length binary vectors (strings of 0s and 1s) or integer arrays that represent the 2D or 3D structure of a molecule. They act like a chemical barcode, transforming a complex chemical structure into a machine-readable format that allows algorithms to calculate molecular similarity, cluster chemical libraries, and train machine learning models) , and structural descriptors. 
- Searching: Run high-performance substructure and molecular similarity comparisons. 
- Database Support: Includes a PostgreSQL cartridge to store and query chemical structures relationally. 
- Integrations: Easily pairs with data science packages like Pandas, KNIME, and Scikit-Learn. 

In [11]:
!pip install rdkit pandas

Defaulting to user installation because normal site-packages is not writeable
  Using cached tzdata-2026.3-py2.py3-none-any.whl.metadata (1.4 kB)
   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   ------------- -------------------------- 3.4/9.8 MB 22.3 MB/s eta 0:00:01
   -------------- ------------------------- 3.7/9.8 MB 9.1 MB/s eta 0:00:01
   ---------------------- ----------------- 5.5/9.8 MB 8.6 MB/s eta 0:00:01
   ------------------------ --------------- 6.0/9.8 MB 7.4 MB/s eta 0:00:01
   ----------------------------- ---------- 7.3/9.8 MB 6.8 MB/s eta 0:00:01
   ---------------------------------- ----- 8.4/9.8 MB 6.6 MB/s eta 0:00:01
   ------------------------------------ --- 8.9/9.8 MB 6.2 MB/s eta 0:00:01
   ------------------------------------- -- 9.2/9.8 MB 5.7 MB/s eta 0:00:01
   ---------------------------------------- 9.8/9.8 MB 4.9 MB/s eta 0:00:00
Using cached tzdata-2026.3-py2.py3-none-any.whl (348 kB)



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Step 2: Download Gefitinib

Download the SDF file from PubChem. Saved [Gefitinib.sdf] File 

# Step 3: Read the Molecule 

In [1]:
from rdkit import Chem

mol = Chem.SDMolSupplier("Gefitinib.sdf")[0]

print(mol) 

# Step 4: Import Descriptor Libraries 
## 1. rdkit.Chem.Lipinski- 
This module evaluates whether a molecule looks like an orally active drug based on structural counting rules. It is primarily used to check compliance with Lipinski's Rule of 5 and similar drug-likeness filters. 

### Common Functions 
- Lipinski.NumHAcceptors(mol): Counts Hydrogen Bond Acceptors (N or O atoms) 
- Lipinski.NumHDonors(mol): Counts Hydrogen Bond Donors (NH or OH groups) 
- Lipinski.HeavyAtomCount(mol): Counts all atoms excluding Hydrogen 
- Lipinski.RotatableBondCount(mol): Counts single bonds that can rotate (indicates molecular flexibility) 

## 2. rdkit.Chem.Crippen- 
This module calculates physical chemistry properties using atom-contribution models developed by Ghose and Crippen. Instead of running heavy simulations, it estimates macroscopic physical properties based on the types of atoms present. 

### Common Functions 
- Crippen.MolLogP(mol): Calculates the Octanol-Water Partition Coefficient (\(\log P\)) And Values above 5 usually mean poor water solubility 
- Crippen.MolMR(mol): Calculates Molar Refractivity. This serves as a proxy for the total volume and polarizability of the molecule. 

## 3. rdkit.Chem.rdMolDescriptors- 
This is a highly optimized C++ backend module. It contains an extensive collection of advanced 2D and 3D molecular descriptors, ranging from surface areas to complex structural shape indexes. 

### Common Functions 
- rdMolDescriptors.CalcTPSA(mol): Topological Polar Surface Area. Compounds with a TPSA greater than \(140\text{ \AA}^2\) usually show poor cell membrane permeability 
- rdMolDescriptors.CalcExactMolWt(mol): Computes the precise isotopic molecular weight 
- rdMolDescriptors.CalcFractionCSP3(mol): Measures molecular complexity by checking the ratio of sp³ hybridized carbons to total carbons. 

In [2]:
from rdkit.Chem import Descriptors
from rdkit.Chem import Lipinski
from rdkit.Chem import Crippen
from rdkit.Chem import rdMolDescriptors 

# Step 5: Calculate Molecular Descriptors 

In [3]:
print("Molecular Weight:", Descriptors.MolWt(mol))

print("LogP:", Crippen.MolLogP(mol))

print("TPSA:", rdMolDescriptors.CalcTPSA(mol))

print("H-Bond Donors:", Lipinski.NumHDonors(mol))

print("H-Bond Acceptors:", Lipinski.NumHAcceptors(mol))

print("Rotatable Bonds:", Lipinski.NumRotatableBonds(mol))

print("Ring Count:", Lipinski.RingCount(mol))

print("Heavy Atoms:", mol.GetNumHeavyAtoms())

print("Fraction Csp3:", rdMolDescriptors.CalcFractionCSP3(mol)) 

Molecular Weight: 446.9100000000002
LogP: 4.275600000000003
TPSA: 68.74000000000001
H-Bond Donors: 1
H-Bond Acceptors: 7
Rotatable Bonds: 8
Ring Count: 4
Heavy Atoms: 31
Fraction Csp3: 0.36363636363636365


# Step 6: Check Drug-Likeness (Lipinski's Rule of Five) 

In [4]:
mw = Descriptors.MolWt(mol)
logp = Crippen.MolLogP(mol)
hbd = Lipinski.NumHDonors(mol)
hba = Lipinski.NumHAcceptors(mol)

print("Lipinski Rule")

print("MW < 500 :", mw < 500)
print("LogP < 5 :", logp < 5)
print("HBD <= 5 :", hbd <= 5)
print("HBA <= 10 :", hba <= 10) 

Lipinski Rule
MW < 500 : True
LogP < 5 : True
HBD <= 5 : True
HBA <= 10 : True


# Step 7: Generate Molecular Fingerprints 

In [5]:
from rdkit.Chem import AllChem

fingerprint = AllChem.GetMorganFingerprintAsBitVect(
    mol,
    radius=2,
    nBits=2048
)

print(fingerprint) 

[01:35:31] DEPRECATION WARNING: please use MorganGenerator


# Step 8: Save Descriptors to a CSV File 

In [6]:
from rdkit.Chem import AllChem

fingerprint = AllChem.GetMorganFingerprintAsBitVect(
    mol,
    radius=2,
    nBits=2048
)

print(fingerprint) 

[01:36:31] DEPRECATION WARNING: please use MorganGenerator


# Step 8: Save Descriptors to a CSV File 

In [12]:
import pandas as pd

data = {
    "Molecular Weight": [Descriptors.MolWt(mol)],
    "LogP": [Crippen.MolLogP(mol)],
    "TPSA": [rdMolDescriptors.CalcTPSA(mol)],
    "HBD": [Lipinski.NumHDonors(mol)],
    "HBA": [Lipinski.NumHAcceptors(mol)],
    "Rotatable Bonds": [Lipinski.NumRotatableBonds(mol)],
    "Ring Count": [Lipinski.RingCount(mol)],
    "Heavy Atoms": [mol.GetNumHeavyAtoms()],
    "Fraction Csp3": [rdMolDescriptors.CalcFractionCSP3(mol)]
}

df = pd.DataFrame(data)

df.to_csv("Gefitinib_Descriptors.csv", index=False)

print(df) 

   Molecular Weight    LogP   TPSA  HBD  HBA  Rotatable Bonds  Ring Count  \
0            446.91  4.2756  68.74    1    7                8           4   

   Heavy Atoms  Fraction Csp3  
0           31       0.363636  


# Step 8: Save Descriptors to a CSV File 

In [15]:
import pandas as pd

data = {
    "Molecular Weight": [Descriptors.MolWt(mol)],
    "LogP": [Crippen.MolLogP(mol)],
    "TPSA": [rdMolDescriptors.CalcTPSA(mol)],
    "HBD": [Lipinski.NumHDonors(mol)],
    "HBA": [Lipinski.NumHAcceptors(mol)],
    "Rotatable Bonds": [Lipinski.NumRotatableBonds(mol)],
    "Ring Count": [Lipinski.RingCount(mol)],
    "Heavy Atoms": [mol.GetNumHeavyAtoms()],
    "Fraction Csp3": [rdMolDescriptors.CalcFractionCSP3(mol)]
}

df = pd.DataFrame(data)

df.to_csv("Gefitinib_Descriptors.csv", index=False)

print(df) 
print("Descriptors saved to Gefitinib_Descriptors.csv") 

   Molecular Weight    LogP   TPSA  HBD  HBA  Rotatable Bonds  Ring Count  \
0            446.91  4.2756  68.74    1    7                8           4   

   Heavy Atoms  Fraction Csp3  
0           31       0.363636  
Descriptors saved to Gefitinib_Descriptors.csv


# Step 9: Generate the 2D Structure Image 

In [20]:
from rdkit import Chem
from rdkit.Chem import Draw

# Read the molecule from the SDF file
mol = Chem.SDMolSupplier("data/Gefitinib.sdf")[0] 

# Generate and save a 2D image
img = Draw.MolToImage(mol, size=(600, 600))
img.save("Output/Gefitinib_2D.png") 

print("2D structure image saved successfully!") 

2D structure image saved successfully!
